In [3]:
import pandas as pd
import joblib
import os

# 1. Load the model you just created (using the same 'hard' path)
model_path = r"C:\Users\VASUNDHARA\OneDrive\Documents\Desktop\geosentinel-plus\outputs\competency_model.pkl"
model = joblib.load(model_path)

# 2. Load the cleaned features from Teammate 2
data_path = r"C:\Users\VASUNDHARA\Downloads\surface_features_clean.csv"
df = pd.read_csv(data_path)

print("Ready to start mapping!")

Ready to start mapping!


In [5]:
# Run this to see your column names
print(df.columns.tolist())

['lat', 'lon', 'elevation', 'slope', 'aspect', 'curvature', 'NDVI', 'rainfall']


In [8]:
# Check the exact order the model expects
expected_features = model.feature_names_in_
print("The model expects this order:", expected_features)

# Now, ensure your features match that exact order
# If the output shows ['RHOB', 'NPHI', 'GR'], use that sequence below:
features_in_order = df_renamed[expected_features]

# Predict
df['competency_score'] = model.predict(features_in_order)
print("Competency scores calculated successfully!")

The model expects this order: ['GR' 'RHOB' 'NPHI']
Competency scores calculated successfully!


In [9]:
print(df[['lat', 'lon', 'competency_score']].head())

         lat        lon  competency_score
0  30.692876  78.715304               0.5
1  30.782188  79.305341               1.0
2  30.384875  78.731183               0.5
3  30.485979  78.971731               0.5
4  30.115728  79.219520               0.5


In [10]:
stable_criteria = (df['slope'] < 10) & (df['NDVI'] > 0.4) & (df['rainfall'] < df['rainfall'].quantile(0.4))
stable_points = df[stable_criteria].copy()
stable_points['landslide_label'] = 0

In [19]:
print("Here are the first 5 stable points:")
print(stable_points.head())


Here are the first 5 stable points:
           lat        lon  elevation     slope      aspect  curvature  \
200  30.489348  79.692290       1942  8.338745  116.216293       71.0   
206  30.202381  79.202287       1606  9.723366  160.857101        7.0   
261  30.387873  79.165742       2497  4.252073  130.781677      -20.0   
305  30.241810  79.465702       2105  9.953707  253.935257       14.0   
377  30.248453  78.844189        726  5.431558  260.197784       77.0   

         NDVI     rainfall  competency_score  landslide_label  
200  0.434680  1049.589946               0.5                0  
206  0.542814  1178.509237               0.5                0  
261  0.597760  1058.607682               0.5                0  
305  0.525840  1082.573669               0.5                0  
377  0.767815  1234.017849               0.5                0  


In [22]:
import numpy as np
df['landslide_label'] = np.nan 

# Fill in the labels for your stable points
df.loc[stable_criteria, 'landslide_label'] = 0

# Save this complete file to the data folder
save_path = r"C:\Users\VASUNDHARA\OneDrive\Documents\Desktop\geosentinel-plus\data\final_training_data.csv"
df.to_csv(save_path, index=False)
print(f"Final training file saved at: {save_path}")

Final training file saved at: C:\Users\VASUNDHARA\OneDrive\Documents\Desktop\geosentinel-plus\data\final_training_data.csv
